# 12. ICVI и многокритериальная оценка сетевой кластеризации

Этот ноутбук предназначен для конкурсной проверки качества разбиений на контрольной точке декабря 2024. Мы оцениваем разные значения `resolution` одновременно в атрибутивном пространстве и в структуре сети.

**Важно:** все варианты Louvain здесь используют одну и ту же реализацию `networkx.community.louvain_communities`, тот же seed `42` и одну и ту же Euclidean-сеть.

In [1]:
import pandas as pd
import numpy as np
import networkx as nx

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score
)
from sklearn.metrics import adjusted_rand_score

print("Библиотеки загружены.")

Библиотеки загружены.


## 1. Загрузка контрольной сети и признаков

Используем те же five-share признаки и ребра, что и в основной Euclidean-сети.

In [2]:
features_dec = pd.read_parquet(
    "../data/processed/features_2024_12.parquet"
)

edges_dec = pd.read_parquet(
    "../data/processed/edges_2024_12.parquet"
)

print("features:", features_dec.shape)
print("edges:", edges_dec.shape)

print("\nКолонки features:")
print(features_dec.columns.tolist())

print("\nКолонки edges:")
print(edges_dec.columns.tolist())

features: (2103, 6)
edges: (13757, 3)

Колонки features:
['territory_id', 'food_share', 'health_share', 'marketplaces_share', 'catering_share', 'transport_share']

Колонки edges:
['source', 'target', 'weight']


In [3]:
feature_cols = [
    "food_share",
    "health_share",
    "marketplaces_share",
    "catering_share",
    "transport_share"
]

X_dec = features_dec[feature_cols].to_numpy()

G_dec = nx.Graph()

G_dec.add_nodes_from(features_dec["territory_id"].tolist())

for row in edges_dec.itertuples(index=False):
    G_dec.add_edge(
        int(row.source),
        int(row.target),
        weight=float(row.weight)
    )

print("Количество узлов:", G_dec.number_of_nodes())
print("Количество рёбер:", G_dec.number_of_edges())
print("Компонент связности:", nx.number_connected_components(G_dec))

Количество узлов: 2103
Количество рёбер: 13757
Компонент связности: 1


## 2. Подключение S_Dbw

S_Dbw используется как дополнительный индекс качества структуры кластеров в пространстве признаков; меньшее значение интерпретируется как более компактное и разделенное разбиение.

In [4]:
from s_dbw import S_Dbw

print("S_Dbw загружен.")

S_Dbw загружен.


In [5]:
resolutions = [0.50, 0.75, 1.00, 1.25, 1.50, 2.00]

partition_results = {}

for resolution in resolutions:
    communities = nx.community.louvain_communities(
        G_dec,
        weight="weight",
        resolution=resolution,
        seed=42
    )

    partition = {}

    for cluster_id, community in enumerate(communities):
        for node in community:
            partition[node] = cluster_id

    partition_results[resolution] = {
        "partition": partition,
        "communities": communities
    }

print("Построено разбиений:", len(partition_results))

for resolution, result in partition_results.items():
    modularity = nx.community.modularity(
        G_dec,
        result["communities"],
        weight="weight"
    )

    print(
        f"resolution={resolution:.2f} | "
        f"кластеров={len(result['communities'])} | "
        f"modularity={modularity:.4f}"
    )

Построено разбиений: 6
resolution=0.50 | кластеров=11 | modularity=0.7991
resolution=0.75 | кластеров=15 | modularity=0.8119
resolution=1.00 | кластеров=17 | modularity=0.8140
resolution=1.25 | кластеров=20 | modularity=0.8145
resolution=1.50 | кластеров=19 | modularity=0.8140
resolution=2.00 | кластеров=29 | modularity=0.7994


## 3. Единая реализация Louvain

Проверяем несколько значений `resolution`. Для каждого сохраняем одновременно dictionary меток и список сообществ, чтобы одной и той же кластеризацией можно было рассчитать все последующие метрики.

In [6]:
attribute_icvi_results = []

for resolution, result in partition_results.items():
    partition = result["partition"]

    labels = np.array([
        partition[int(territory_id)]
        for territory_id in features_dec["territory_id"]
    ])

    n_clusters = len(np.unique(labels))

    silhouette = silhouette_score(
        X_dec,
        labels,
        metric="euclidean"
    )

    calinski = calinski_harabasz_score(
        X_dec,
        labels
    )

    s_dbw = S_Dbw(
        X_dec,
        labels,
        method="Tong",
        alg_noise="bind",
        centr="mean",
        nearest_centr=True,
        metric="euclidean"
    )

    attribute_icvi_results.append({
        "resolution": resolution,
        "n_clusters": n_clusters,
        "silhouette": silhouette,
        "calinski_harabasz": calinski,
        "s_dbw": s_dbw,
        "modularity": nx.community.modularity(
            G_dec,
            result["communities"],
            weight="weight"
        )
    })

attribute_icvi_df = pd.DataFrame(attribute_icvi_results)

attribute_icvi_df = attribute_icvi_df.sort_values(
    "resolution"
).reset_index(drop=True)

attribute_icvi_df.round(6)

,resolution,n_clusters,silhouette,calinski_harabasz,s_dbw,modularity
0,0.50,11,0.178519,877.296293,0.409699,0.799149
1,0.75,15,0.191845,892.947191,0.368383,0.811850
2,1.00,17,0.170096,852.564817,0.347601,0.814024
3,1.25,20,0.163593,879.100115,0.321043,0.814475
4,1.50,19,0.178424,914.598833,0.342515,0.813972
5,2.00,29,0.114000,623.715928,0.281018,0.799360


In [7]:
node_ids = features_dec["territory_id"].astype(int).tolist()

A = nx.to_numpy_array(
    G_dec,
    nodelist=node_ids,
    weight="weight",
    dtype=float
)

network_icvi_results = []

for resolution, result in partition_results.items():
    partition = result["partition"]
    communities = result["communities"]

    labels = np.array([
        partition[int(territory_id)]
        for territory_id in features_dec["territory_id"]
    ])

    cluster_ids = sorted(np.unique(labels))
    K = len(cluster_ids)

    isolabilities = []

    for cluster_id in cluster_ids:
        idx = labels == cluster_id
        not_idx = ~idx

        internal_weight = A[np.ix_(idx, idx)].sum()
        external_weight = A[np.ix_(idx, not_idx)].sum()

        denominator = internal_weight + external_weight

        if denominator > 0:
            isolability = internal_weight / denominator
        else:
            isolability = 0.0

        isolabilities.append(isolability)

    AVI = np.mean(isolabilities)

    unifiabilities = []

    for k in cluster_ids:
        idx_k = labels == k
        external_k = A[np.ix_(idx_k, ~idx_k)].sum()

        for l in cluster_ids:
            if l == k:
                continue

            idx_l = labels == l

            between_kl = A[np.ix_(idx_k, idx_l)].sum()
            external_l = A[np.ix_(idx_l, ~idx_l)].sum()

            denominator = (
                external_k
                + external_l
                - between_kl
            )

            if denominator > 0:
                unifiability = between_kl / denominator
            else:
                unifiability = 0.0

            unifiabilities.append(unifiability)

    AVU = np.sum(unifiabilities) / K

    ANUI = AVI / (1 + AVI * AVU)

    cluster_factors = []

    for k in cluster_ids:
        idx_k = labels == k
        not_idx_k = ~idx_k

        internal_weight = (
            A[np.ix_(idx_k, idx_k)].sum() / 2
        )

        external_weight = A[np.ix_(idx_k, not_idx_k)].sum()

        if internal_weight > 0:
            cluster_factor = (
                internal_weight
                / (
                    internal_weight
                    + 0.5 * external_weight
                )
            )
        else:
            cluster_factor = 0.0

        cluster_factors.append(cluster_factor)

    MQ = np.sum(cluster_factors)
    MQ_per_cluster = MQ / K

    network_icvi_results.append({
        "resolution": resolution,
        "n_clusters": K,
        "AVI": AVI,
        "AVU": AVU,
        "ANUI": ANUI,
        "MQ": MQ,
        "MQ_per_cluster": MQ_per_cluster
    })

network_icvi_df = pd.DataFrame(network_icvi_results)

network_icvi_df = network_icvi_df.sort_values(
    "resolution"
).reset_index(drop=True)

network_icvi_df.round(6)

,resolution,n_clusters,AVI,AVU,ANUI,MQ,MQ_per_cluster
0,0.50,11,0.927047,0.487287,0.638577,10.197518,0.927047
1,0.75,15,0.904116,0.492702,0.625487,13.561746,0.904116
2,1.00,17,0.892864,0.489110,0.621465,15.178696,0.892864
3,1.25,20,0.879399,0.481299,0.617879,17.587989,0.879399
4,1.50,19,0.886361,0.489500,0.618158,16.840858,0.886361
5,2.00,29,0.848393,0.500478,0.595530,24.603404,0.848393


In [8]:
icvi_results = attribute_icvi_df.merge(
    network_icvi_df,
    on=["resolution", "n_clusters"],
    how="inner"
)

icvi_results["ch_per_node"] = (
    icvi_results["calinski_harabasz"] / len(X_dec)
)

icvi_results = icvi_results[
    [
        "resolution",
        "n_clusters",
        "silhouette",
        "calinski_harabasz",
        "ch_per_node",
        "s_dbw",
        "AVI",
        "AVU",
        "ANUI",
        "MQ",
        "modularity"
    ]
]

icvi_results.round(6)

,resolution,n_clusters,silhouette,calinski_harabasz,ch_per_node,s_dbw,AVI,AVU,ANUI,MQ,modularity
0,0.50,11,0.178519,877.296293,0.417164,0.409699,0.927047,0.487287,0.638577,10.197518,0.799149
1,0.75,15,0.191845,892.947191,0.424606,0.368383,0.904116,0.492702,0.625487,13.561746,0.811850
2,1.00,17,0.170096,852.564817,0.405404,0.347601,0.892864,0.489110,0.621465,15.178696,0.814024
3,1.25,20,0.163593,879.100115,0.418022,0.321043,0.879399,0.481299,0.617879,17.587989,0.814475
4,1.50,19,0.178424,914.598833,0.434902,0.342515,0.886361,0.489500,0.618158,16.840858,0.813972
5,2.00,29,0.114000,623.715928,0.296584,0.281018,0.848393,0.500478,0.595530,24.603404,0.799360


## 4. Атрибутивные ICVI

Для каждой настройки считаем Silhouette, Calinski-Harabasz и S_Dbw. Все три индекса рассчитываются по одним и тем же пяти экономическим долям.

In [9]:
ranking_df = icvi_results[
    [
        "resolution",
        "n_clusters",
        "silhouette",
        "calinski_harabasz",
        "s_dbw",
        "AVI",
        "AVU",
        "ANUI",
        "MQ",
        "modularity"
    ]
].copy()

metric_directions = {
    "silhouette": True,
    "calinski_harabasz": True,
    "s_dbw": False,
    "AVI": True,
    "AVU": False,
    "ANUI": True,
    "MQ": True,
    "modularity": True
}

rank_columns = []

for metric, higher_is_better in metric_directions.items():
    rank_col = f"{metric}_rank"
    ranking_df[rank_col] = ranking_df[metric].rank(
        ascending=not higher_is_better,
        method="min"
    ).astype(int)
    rank_columns.append(rank_col)

ranking_df["mean_rank_diagnostic"] = (
    ranking_df[rank_columns].mean(axis=1)
)

ranking_df["n_top2"] = (
    ranking_df[rank_columns].le(2).sum(axis=1)
)

ranking_summary = ranking_df[
    ["resolution", "n_clusters", *rank_columns,
     "mean_rank_diagnostic", "n_top2"]
].sort_values(
    ["mean_rank_diagnostic", "n_top2"],
    ascending=[True, False]
)

display(ranking_summary.round(3))


,resolution,n_clusters,silhouette_rank,calinski_harabasz_rank,s_dbw_rank,AVI_rank,AVU_rank,ANUI_rank,MQ_rank,modularity_rank,mean_rank_diagnostic,n_top2
3,1.25,20,5,3,2,5,1,5,2,1,3.000,4
4,1.50,19,3,1,3,4,4,4,3,3,3.125,1
1,0.75,15,1,2,5,2,5,2,5,4,3.250,4
0,0.50,11,2,4,6,1,2,1,6,6,3.500,4
2,1.00,17,4,5,4,3,3,3,4,2,3.500,1
5,2.00,29,6,6,1,6,6,6,1,5,4.625,2


## 5. Сетевые ICVI

Дополнительно оцениваем внутреннюю и внешнюю связность кластеров через реализованные в проекте сетевые показатели AVI, AVU, ANUI и MQ.

In [10]:
from itertools import combinations

stability_all = []
seeds = [1, 42, 100, 2024, 999]

for resolution in resolutions:
    partitions_by_seed = {}

    for seed in seeds:
        communities = nx.community.louvain_communities(
            G_dec,
            weight="weight",
            resolution=resolution,
            seed=seed
        )

        labels = {}
        for cluster_id, community in enumerate(communities):
            for node in community:
                labels[node] = cluster_id

        partitions_by_seed[seed] = np.array([
            labels[int(territory_id)]
            for territory_id in features_dec["territory_id"]
        ])

    ari_values = []

    for seed_a, seed_b in combinations(partitions_by_seed.keys(), 2):
        ari_values.append(
            adjusted_rand_score(
                partitions_by_seed[seed_a],
                partitions_by_seed[seed_b]
            )
        )

    stability_all.append({
        "resolution": resolution,
        "ARI_mean": np.mean(ari_values),
        "ARI_min": np.min(ari_values),
        "ARI_max": np.max(ari_values)
    })

stability_all_df = pd.DataFrame(stability_all)
display(stability_all_df.round(4))


,resolution,ARI_mean,ARI_min,ARI_max
0,0.50,0.6501,0.5426,0.7955
1,0.75,0.6392,0.5676,0.7011
2,1.00,0.6529,0.6101,0.7049
3,1.25,0.6856,0.6017,0.7463
4,1.50,0.6187,0.5461,0.7220
5,2.00,0.6753,0.5748,0.7607


## 6. Объединение атрибутивных и сетевых показателей

Получаем единую таблицу, где для каждого `resolution` собраны сетевые и атрибутивные характеристики.

In [11]:
icvi_results.to_csv(
    "../data/processed/icvi_results_dec2024.csv",
    index=False
)

ranking_summary.to_csv(
    "../data/processed/icvi_ranking_dec2024.csv",
    index=False
)

stability_all_df.to_csv(
    "../data/processed/louvain_stability_dec2024.csv",
    index=False
)

print("ICVI-результаты сохранены.")


ICVI-результаты сохранены.


### Важное замечание по сетевым показателям

`MQ_per_cluster` сохранялся в промежуточном расчете как служебная величина, но не используется как независимый критерий в многокритериальном ранжировании, поскольку в текущей реализации он совпадает с `AVI`.